# Build Metadata Store — data.wa.gov

Crawls **every dataset** on the Washington State Open Data portal and captures what this project trains on: every **column**'s name, type, **per-column statistics** (completeness, null count, distinct-value count, and value range/aggregates), up to 10 distinct **sample values**, and the human-written **column description** that serves as the gold target.

The dataset's own **description** is captured too, but only as *context* shown to the model when it writes a column description (it mirrors the `{datasetDescription}` slot in the production tool's `prompts/column.md`) — it is **not** a training target. This project trains a single task: **column description generation**.

Datasets are discovered automatically through the **Socrata Discovery API** — there is no hand-curated list. The portal:

- `data.wa.gov` — Washington State Open Data

**Columns with empty descriptions are not used.** Many open-data assets ship with undocumented columns. A dataset is only written to the store if **at least one column** carries a usable description; columns with empty descriptions stay blank and are skipped downstream. This keeps empty targets out of the train / validation / test splits the finetune and eval notebooks build.

**Datasets with fewer than `MIN_ROWS` (default 10) rows are also dropped**, since columns need enough rows to harvest distinct sample values from.

**Sponsor "golden" datasets are tagged, not curated by hand.** If `GOLDEN_XLSX` points at the sponsor's allowlist workbook, only its **UID column** is read (the stored descriptions are outdated, so the latest metadata is fetched live like every other dataset). Those UIDs are always fetched, **exempt from the drop filters**, and written with `"golden": true` — `finetune_descriptions.ipynb` pins them to the held-out test split and `evaluate_descriptions.ipynb` reports a gold benchmark on them.

Outputs (resumable — re-running skips datasets already saved):

- `metadata_store/<uid>.json` — one record per dataset
- `all_metadata.json` — combined `{uid: record}` consumed by `finetune_descriptions.ipynb` / `evaluate_descriptions.ipynb`
- `metadata_store/_index.json` — run summary (per-dataset status, per-portal counts, drops, errors)

Standard library (`json`, `urllib`) for the crawl; `openpyxl` is needed only to read the golden allowlist (install cell below). Data comes from the Socrata Discovery API (`api.us.socrata.com/api/catalog/v1`) and the per-dataset APIs (`/api/views/<uid>.json` for name, descriptions, and per-column `cachedContents` statistics; `/resource/<uid>.json` for sample rows).

In [ ]:
# Only extra dependency: openpyxl, to read the sponsor "golden" allowlist xlsx (its UID column).
# The crawl itself is pure standard library. Skip this if you set GOLDEN_XLSX = None below.
%pip install -q openpyxl

## 1. Configuration

In [ ]:
import json
import re
import time
import urllib.parse
import urllib.request
import urllib.error
from datetime import datetime, timezone
from pathlib import Path

# ---- Portal to crawl (Socrata API host) ----
# Every *dataset*-type asset on the domain is discovered automatically (no CSV).
PORTALS = [
    "data.wa.gov",  # Washington State Open Data
]
MAX_DATASETS_PER_DOMAIN = None  # None = every dataset on the portal; set an int to cap (e.g. 1250 for a smoke test)

# ---- Discovery API (catalog) ----
CATALOG_API = "https://api.us.socrata.com/api/catalog/v1"
CATALOG_PAGE_SIZE = 1000  # results per scroll page

# ---- Outputs ----
OUT_DIR = Path("metadata_store")  # per-dataset JSON files land here
COMBINED_PATH = Path("all_metadata.json")
INDEX_PATH = OUT_DIR / "_index.json"

# ---- What counts as a usable description (never train/test/val on empties) ----
# This project trains ONE task: column descriptions. A dataset is kept only if at
# least one of its columns carries a description at least MIN_COLUMN_DESC_CHARS long
# (matches the threshold in finetune_descriptions.ipynb / evaluate_descriptions.ipynb).
# The dataset's own description is kept as *context* for the column prompt, not as a
# target; MIN_DATASET_DESC_CHARS only gates whether it is long enough to show.
MIN_DATASET_DESC_CHARS = (
    40  # dataset description shorter than this is treated as missing (context only)
)
MIN_COLUMN_DESC_CHARS = 10  # column description shorter than this is treated as missing
DROP_DATASETS_WITHOUT_USABLE_DESC = (
    True  # skip datasets with no usable column-description
)
MIN_ROWS = (
    10  # skip datasets with fewer rows than this (need >= this many sample values)
)

# ---- Fetch behavior ----
SAMPLES_PER_COLUMN = 10  # distinct non-null example values to keep per column
ROWS_TO_SCAN = 100  # rows pulled per dataset to harvest those samples
REQUEST_TIMEOUT = 60  # seconds per HTTP request
SLEEP_BETWEEN = 0.3  # politeness pause between datasets (seconds)
APP_TOKEN = (
    ""  # optional Socrata app token (raises rate limits); recommended for a full crawl
)
SKIP_COMPUTED = True  # drop Socrata system/computed columns (fieldName starts with ":")
OVERWRITE = False  # False = resume (skip datasets already saved); True = re-fetch all

# ---- Sponsor "golden" allowlist (eval anchor) ----
# A Socrata catalog export the sponsor flagged as strong-metadata examples. We use
# ONLY its UID column (the stored descriptions are outdated): these UIDs are always
# fetched, exempted from the drop filters, and tagged "golden": true in the store.
# finetune_descriptions.ipynb pins them to the test split; evaluate reports a gold
# benchmark on them. Set to None to disable.
GOLDEN_XLSX = Path(
    "FourMusketeersCapstone_DatasetsWithSolidMetadata(DataWA)_20260227.xlsx"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Portals: {', '.join(PORTALS)}")
print(f"Output dir: {OUT_DIR.resolve()}")

In [ ]:
# --- Run environment: Colab (Drive) OR local / remote GPU server ---
# On Colab the filesystem is empty, so we mount Drive and re-root every output under
# PROJECT_DIR. Off Colab (your own machine or a remote GPU box) everything stays on
# the local disk; point the run at a folder by exporting PROJECT_DIR, e.g.
#   export PROJECT_DIR=/data/metadata-finetune   (defaults to ".").
# Datasets are discovered live from the Socrata Discovery API, so there is nothing to upload.
import os

try:
    import google.colab  # are we on Colab?

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = IN_COLAB  # set False to keep everything on Colab's local /content disk
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path(
        os.environ.get("PROJECT_DIR", "/content/drive/MyDrive/MetadataFineTuningTool")
    )  # <-- your Drive project folder
else:
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "."))
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# re-root outputs under PROJECT_DIR (overrides the Configuration cell)
OUT_DIR = PROJECT_DIR / "metadata_store"
OUT_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_PATH = PROJECT_DIR / "all_metadata.json"
INDEX_PATH = OUT_DIR / "_index.json"

# the golden allowlist xlsx lives alongside the project (upload it to PROJECT_DIR on Colab)
if GOLDEN_XLSX is not None:
    GOLDEN_XLSX = PROJECT_DIR / Path(GOLDEN_XLSX).name

print("Store ->", PROJECT_DIR)

## 2. Helper functions

HTTP + JSON, HTML stripping, timestamp conversion, column filtering, and sample collection — shared by both the discovery crawl and the per-dataset fetch.

In [ ]:
def _request(url):
    """GET a URL and parse JSON, sending the app token if configured."""
    req = urllib.request.Request(url)
    if APP_TOKEN:
        req.add_header("X-App-Token", APP_TOKEN)
    with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT) as resp:
        return json.load(resp)


def strip_html(s):
    """Remove HTML tags and collapse whitespace in a metadata string."""
    if not s:
        return ""
    s = re.sub(r"<[^>]+>", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def epoch_to_iso(ts):
    """Convert a Socrata epoch-seconds timestamp to ISO 8601 (UTC)."""
    if not ts:
        return None
    try:
        return datetime.fromtimestamp(int(ts), tz=timezone.utc).isoformat()
    except (ValueError, TypeError, OSError):
        return ts


def is_real_column(col):
    """True for user-facing data columns; skips system/computed (':' / ':@computed_*')."""
    field = col.get("fieldName") or ""
    if not field:
        return False
    if SKIP_COMPUTED and field.startswith(":"):
        return False
    return True


def _sample_value(v):
    """Make a row value JSON-friendly (location/point cells arrive as dicts)."""
    if isinstance(v, (dict, list)):
        return json.dumps(v, ensure_ascii=False, default=str)
    return v


def collect_samples(field, rows, cached_top, k):
    """Up to k distinct non-null values for `field`, from rows then cachedContents.top."""
    samples, seen = [], set()

    def add(raw):
        if raw is None or raw == "":
            return False
        val = _sample_value(raw)
        key = val if isinstance(val, str) else json.dumps(val, default=str)
        if key in seen:
            return False
        seen.add(key)
        samples.append(val)
        return True

    for r in rows:
        if field in r and add(r[field]) and len(samples) >= k:
            return samples
    for t in cached_top or []:  # fallback when rows didn't yield enough
        if add(t.get("item")) and len(samples) >= k:
            break
    return samples


def _to_int(v):
    """Best-effort int from Socrata's string-typed counts; None if not parseable."""
    try:
        return int(v)
    except (TypeError, ValueError):
        return None


def column_stats(col):
    """Per-column statistics from Socrata `cachedContents`, mirroring prompts/column.md.

    Socrata caches a profile per column: non_null / null counts, distinct
    `cardinality`, the `smallest`/`largest` observed values, and numeric
    `average`/`sum`. We surface completeness (non_null / row_count) so the column
    prompt can fill {nonNullCount}, {rowCount}, {completenessPercent}, {nullCount},
    and {columnStats}. Returns a compact dict with empty/None fields dropped.
    """
    cc = col.get("cachedContents") or {}
    non_null = _to_int(cc.get("non_null"))
    null = _to_int(cc.get("null"))
    row_count = None
    if non_null is not None or null is not None:
        row_count = (non_null or 0) + (null or 0)
    completeness = (
        round(100 * non_null / row_count, 1)
        if non_null is not None and row_count
        else None
    )
    stats = {
        "non_null": non_null,
        "null": null,
        "row_count": row_count,
        "completeness_pct": completeness,
        "cardinality": _to_int(cc.get("cardinality")),
        "smallest": cc.get("smallest"),
        "largest": cc.get("largest"),
        "average": cc.get("average"),
        "sum": cc.get("sum"),
    }
    return {k: v for k, v in stats.items() if v not in (None, "")}


def load_golden_uids(path):
    """Read the sponsor 'golden' allowlist xlsx and return the set of unique UIDs.

    Only the UID column is used — the xlsx's stored descriptions are outdated, so
    the latest metadata is fetched live from the portal like every other dataset.
    Returns an empty set (with a note) if the file or openpyxl is unavailable.
    """
    if not path:
        return set()
    path = Path(path)
    if not path.exists():
        print(f"Golden xlsx not found at {path} — no datasets will be tagged 'golden'.")
        return set()
    try:
        from openpyxl import load_workbook
    except ImportError:
        print(
            "openpyxl not installed (run the install cell) — skipping golden tagging."
        )
        return set()
    wb = load_workbook(path, read_only=True, data_only=True)
    rows = wb.active.iter_rows(values_only=True)
    header = [str(h).strip().lower() if h is not None else "" for h in next(rows, [])]
    uid_col = header.index("uid") if "uid" in header else 0  # fall back to first column
    return {
        str(r[uid_col]).strip()
        for r in rows
        if uid_col < len(r) and r[uid_col] not in (None, "")
    }


GOLDEN_UIDS = load_golden_uids(GOLDEN_XLSX)
print(f"Golden allowlist: {len(GOLDEN_UIDS)} unique UIDs")

## 3. Discover every dataset on data.wa.gov

Pages through the **Socrata Discovery API** for each portal in `PORTALS`, collecting the UID, name, and domain of every `dataset`-type asset. Uses scroll-cursor pagination (`scroll_id` = the previous page's last result id), which has no 10,000-row offset ceiling, so it reaches *all* datasets even on a large portal. Set `MAX_DATASETS_PER_DOMAIN` to cap the crawl while testing.

In [ ]:
def discover_datasets(domain, max_n=None, page_size=CATALOG_PAGE_SIZE):
    """Every `dataset`-type asset on a Socrata domain, via the Discovery API.

    Scroll-cursor pagination (scroll_id = the previous page's last resource id) has no
    10k-offset ceiling, so it reaches all datasets. Returns [{uid, name, domain}].
    """
    out, scroll_id = [], ""
    while True:
        params = {
            "domains": domain,
            "search_context": domain,
            "only": "dataset",
            "limit": page_size,
        }
        if scroll_id:
            params["scroll_id"] = scroll_id
        page = _request(CATALOG_API + "?" + urllib.parse.urlencode(params))
        results = page.get("results") or []
        if not results:
            break
        for r in results:
            res = r.get("resource") or {}
            uid = res.get("id")
            if not uid:
                continue
            out.append(
                {
                    "uid": uid,
                    "name": res.get("name") or "",
                    "domain": (r.get("metadata") or {}).get("domain") or domain,
                }
            )
            if max_n and len(out) >= max_n:
                return out
        scroll_id = (results[-1].get("resource") or {}).get("id") or ""
        if not scroll_id:
            break
        time.sleep(SLEEP_BETWEEN)
    return out


# Crawl every portal, de-duplicating UIDs (a dataset can be federated onto >1 domain).
datasets, seen = [], set()
for domain in PORTALS:
    found = discover_datasets(domain, max_n=MAX_DATASETS_PER_DOMAIN)
    new = 0
    for d in found:
        if d["uid"] in seen:
            continue
        seen.add(d["uid"])
        datasets.append(d)
        new += 1
    print(f"{domain}: discovered {len(found)} datasets ({new} new)")

print(f"\nTotal unique datasets to fetch: {len(datasets)}")
for d in datasets[:5]:
    print(f"  {d['domain']}  {d['uid']}  {d['name'][:50]}")
print("  ...")


# Ensure every golden UID is in the crawl even if MAX_DATASETS_PER_DOMAIN caps it.
_present = {d["uid"] for d in datasets}
_added = 0
for uid in sorted(GOLDEN_UIDS):
    if uid not in _present:
        datasets.append({"uid": uid, "name": "", "domain": PORTALS[0]})
        _present.add(uid)
        _added += 1
if GOLDEN_UIDS:
    print(
        f"Golden UIDs ensured present: +{_added} added ({len(datasets)} total to fetch)"
    )

## 4. Build one record per dataset

For each discovered dataset, fetch its view (name, description, column definitions, and per-column `cachedContents` statistics) and a sample of rows **from that dataset's own domain**, then assemble the stored record. Each column keeps its `stats` (completeness, null count, distinct-value count, value range/aggregates) alongside its samples and gold description. `record_has_usable_description` decides whether the result is worth keeping — **at least one column with a usable description** (the dataset's own description is context-only, not a target).

In [ ]:
def fetch_view(uid, domain):
    """Dataset-level metadata: name, description, and column definitions."""
    return _request(f"https://{domain}/api/views/{uid}.json")


def fetch_rows(uid, domain, limit):
    """Up to `limit` sample rows (keyed by column fieldName)."""
    return _request(f"https://{domain}/resource/{uid}.json?$limit={limit}")


def build_record(uid, domain, fallback_name=""):
    """Assemble the stored record for one dataset on `domain`."""
    view = fetch_view(uid, domain)

    row_error = None
    try:
        rows = fetch_rows(uid, domain, ROWS_TO_SCAN)
    except Exception as e:  # keep the dataset even if row fetch fails
        rows, row_error = [], f"{type(e).__name__}: {e}"

    columns = []
    for col in view.get("columns", []) or []:
        if not is_real_column(col):
            continue
        field = col.get("fieldName")
        cached_top = (col.get("cachedContents") or {}).get("top")
        columns.append(
            {
                "name": col.get("name"),
                "field": field,
                "type": col.get("dataTypeName"),
                "description": strip_html(col.get("description")),
                "stats": column_stats(col),
                "samples": collect_samples(field, rows, cached_top, SAMPLES_PER_COLUMN),
            }
        )

    # Dataset row count from the cached per-column profile (non_null + null is the
    # same total for every column); falls back to None when no column carries it.
    dataset_row_count = (
        max((c["stats"].get("row_count") or 0 for c in columns), default=0) or None
    )

    meta = view.get("metadata") or {}
    record = {
        "uid": uid,
        "name": view.get("name") or fallback_name,
        "description": strip_html(view.get("description")),
        "attribution": view.get("attribution"),
        "category": view.get("category"),
        "tags": view.get("tags") or [],
        "row_label": meta.get("rowLabel") if isinstance(meta, dict) else None,
        "row_count": dataset_row_count,
        "column_count": len(columns),
        "created_at": epoch_to_iso(view.get("createdAt")),
        "updated_at": epoch_to_iso(
            view.get("rowsUpdatedAt") or view.get("viewLastModified")
        ),
        "domain": domain,
        "page_url": f"https://{domain}/d/{uid}",
        "api_endpoint": f"https://{domain}/resource/{uid}.json",
        "rows_scanned": len(rows),
        "columns": columns,
        "fetched_at": datetime.now(timezone.utc).isoformat(),
    }
    if row_error:
        record["row_fetch_error"] = row_error
    return record


def has_usable_dataset_desc(record):
    """True if the dataset's own description is long enough to show as context."""
    return len((record.get("description") or "").strip()) >= MIN_DATASET_DESC_CHARS


def usable_column_count(record):
    """How many columns carry a description long enough to learn from."""
    return sum(
        1
        for c in record.get("columns") or []
        if len((c.get("description") or "").strip()) >= MIN_COLUMN_DESC_CHARS
    )


def record_has_usable_description(record):
    """Keep a dataset only if >=1 column has a usable description (column-desc task)."""
    return usable_column_count(record) > 0

## 5. Fetch every dataset

Writes one JSON file per dataset as it goes. Safe to re-run — already-saved datasets are skipped unless `OVERWRITE = True`. Datasets with **fewer than `MIN_ROWS` rows** are skipped, since they can't supply enough distinct sample values. Datasets with **no usable column-description** are also skipped (not written) when `DROP_DATASETS_WITHOUT_USABLE_DESC` is set, so empty targets never reach the splits. Transient network errors are caught and recorded so one bad dataset doesn't halt the run.

As it runs, the loop also tallies the **share of columns that ship with an empty (blank) description** across every dataset it touches this run — including the ones dropped above (a no-description dataset counts as 100% empty). The final summary prints `empty_desc_columns / total_columns` as a percentage, and each written dataset's `_index.json` entry records its own `empty_columns` count alongside `usable_columns`. This is the documentation gap the fine-tuned model is meant to fill.

In [ ]:
index, errors = [], []
n_dropped = 0
n_dropped_rows = 0
total_columns = (
    0  # every real column we touched this run (cached + fetched, incl. skipped)
)
empty_desc_columns = 0  # ...of those, how many ship with a blank description


def _empty_desc_count(rec):
    """Columns in `rec` whose description is blank — the gap the model fills in."""
    return sum(
        1 for c in rec.get("columns") or [] if not (c.get("description") or "").strip()
    )


for i, d in enumerate(datasets, 1):
    uid, domain = d["uid"], d["domain"]
    out_path = OUT_DIR / f"{uid}.json"

    if out_path.exists() and not OVERWRITE:
        rec = json.loads(out_path.read_text(encoding="utf-8"))
        total_columns += len(rec.get("columns") or [])
        empty_desc_columns += _empty_desc_count(rec)
        index.append(
            {
                "uid": uid,
                "domain": rec.get("domain", domain),
                "name": rec.get("name"),
                "column_count": rec.get("column_count"),
                "status": "cached",
            }
        )
        print(f"[{i:>4}/{len(datasets)}] {uid}  cached")
        continue

    try:
        rec = build_record(uid, domain, d["name"])
        is_golden = uid in GOLDEN_UIDS
        rec["golden"] = is_golden
        # Count every column we fetched toward the empty-description rate up front, BEFORE
        # the drop branches below — so datasets skipped for too few rows or no usable
        # description still contribute (a no-description dataset is 100% empty and counts).
        n_empty = _empty_desc_count(rec)
        total_columns += len(rec.get("columns") or [])
        empty_desc_columns += n_empty

        # Drop datasets with too few rows to yield enough sample values.
        # rows_scanned is min(total rows, ROWS_TO_SCAN), or 0 if the row fetch failed —
        # either way we can't guarantee MIN_ROWS samples, so the dataset is skipped.
        if (
            rec["rows_scanned"] < MIN_ROWS and not is_golden
        ):  # golden datasets are kept regardless
            n_dropped_rows += 1
            index.append(
                {
                    "uid": uid,
                    "domain": domain,
                    "name": rec.get("name"),
                    "column_count": rec.get("column_count"),
                    "empty_columns": n_empty,  # still counted toward the empty-desc rate
                    "rows_scanned": rec["rows_scanned"],
                    "status": "skipped_too_few_rows",
                }
            )
            print(
                f"[{i:>4}/{len(datasets)}] {uid}  skip ({rec['rows_scanned']} rows < {MIN_ROWS})  "
                f"{(rec.get('name') or '')[:42]}"
            )
            time.sleep(SLEEP_BETWEEN)
            continue

        # Drop datasets with no usable description so empties never enter the splits.
        if (
            DROP_DATASETS_WITHOUT_USABLE_DESC
            and not record_has_usable_description(rec)
            and not is_golden  # golden datasets are kept regardless
        ):
            n_dropped += 1
            index.append(
                {
                    "uid": uid,
                    "domain": domain,
                    "name": rec.get("name"),
                    "column_count": rec.get("column_count"),
                    "empty_columns": n_empty,  # still counted toward the empty-desc rate
                    "status": "skipped_no_description",
                }
            )
            print(
                f"[{i:>4}/{len(datasets)}] {uid}  skip (no usable description)  "
                f"{(rec.get('name') or '')[:42]}"
            )
            time.sleep(SLEEP_BETWEEN)
            continue

        out_path.write_text(
            json.dumps(rec, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
        )
        index.append(
            {
                "uid": uid,
                "domain": domain,
                "name": rec["name"],
                "column_count": rec["column_count"],
                "usable_columns": usable_column_count(rec),
                "empty_columns": n_empty,
                "has_dataset_desc": has_usable_dataset_desc(rec),
                "golden": is_golden,
                "rows_scanned": rec["rows_scanned"],
                "status": "ok",
            }
        )
        note = "  (row fetch failed)" if rec.get("row_fetch_error") else ""
        print(
            f"[{i:>4}/{len(datasets)}] {uid}  ok  {rec['column_count']:>2} cols, "
            f"{rec['rows_scanned']:>3} rows  {(rec['name'] or '')[:42]}{note}"
        )
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        errors.append({"uid": uid, "domain": domain, "error": msg})
        index.append(
            {"uid": uid, "domain": domain, "name": d["name"], "status": "error"}
        )
        print(f"[{i:>4}/{len(datasets)}] {uid}  ERROR  {msg}")

    time.sleep(SLEEP_BETWEEN)

n_ok = sum(1 for x in index if x["status"] == "ok")
n_cached = sum(1 for x in index if x["status"] == "cached")
pct_empty = (100 * empty_desc_columns / total_columns) if total_columns else 0.0
print(
    f"\nDone. ok={n_ok}  cached={n_cached}  "
    f"skipped_no_description={n_dropped}  skipped_too_few_rows={n_dropped_rows}  "
    f"errors={len(errors)}"
)
print(
    f"Columns with empty description: {empty_desc_columns:,}/{total_columns:,} "
    f"({pct_empty:.1f}%)  — includes skipped datasets; excludes fetch errors"
)

## 6. Write the combined file + run index

In [ ]:
records = {}
for p in sorted(OUT_DIR.glob("*.json")):
    if p.name == "_index.json":
        continue
    rec = json.loads(p.read_text(encoding="utf-8"))
    rec["golden"] = (
        rec.get("uid", p.stem) in GOLDEN_UIDS
    )  # keep the golden flag in sync with the allowlist
    records[p.stem] = rec

COMBINED_PATH.write_text(
    json.dumps(records, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
)

# how many datasets actually landed in the store, per portal
count_by_domain = {}
for r in records.values():
    dom = r.get("domain", "?")
    count_by_domain[dom] = count_by_domain.get(dom, 0) + 1

n_golden = sum(1 for r in records.values() if r.get("golden"))

INDEX_PATH.write_text(
    json.dumps(
        {
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "portals": PORTALS,
            "max_datasets_per_domain": MAX_DATASETS_PER_DOMAIN,
            "samples_per_column": SAMPLES_PER_COLUMN,
            "rows_scanned_per_dataset": ROWS_TO_SCAN,
            "min_rows": MIN_ROWS,
            "min_dataset_desc_chars": MIN_DATASET_DESC_CHARS,
            "min_column_desc_chars": MIN_COLUMN_DESC_CHARS,
            "count": len(records),
            "count_golden": n_golden,
            "count_by_domain": count_by_domain,
            "datasets": index,
            "errors": errors,
        },
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

print(f"Wrote {COMBINED_PATH}  ({len(records)} datasets, {n_golden} golden)")
print(f"By portal: {count_by_domain}")
print(f"Wrote {INDEX_PATH}")
if errors:
    print(f"\n{len(errors)} dataset(s) errored — see _index.json:")
    for e in errors[:20]:
        print(f"  {e['uid']}: {e['error']}")

## 7. Fetch the data later

Two convenience loaders. `load_dataset` reads a single dataset's JSON; `load_all` reads the combined file.

In [ ]:
def load_dataset(uid, store_dir=OUT_DIR):
    """Load one dataset's metadata record by UID."""
    return json.loads((Path(store_dir) / f"{uid}.json").read_text(encoding="utf-8"))


def load_all(path=COMBINED_PATH):
    """Load every dataset record as a {uid: record} dict."""
    return json.loads(Path(path).read_text(encoding="utf-8"))


# --- quick verification: show the first stored dataset that has a described column ---
_all = load_all()
_uid = next(
    (
        u
        for u, r in _all.items()
        if any((c.get("description") or "").strip() for c in r.get("columns") or [])
    ),
    next(iter(_all), None),
)
if _uid:
    rec = _all[_uid]
    print(
        f"{rec['name']}  [{rec.get('domain')}]  "
        f"({rec['column_count']} columns, {rec.get('row_count') or '?'} rows)"
    )
    print(f"description (context): {(rec.get('description') or '(none)')[:160]}...\n")
    for c in rec["columns"][:4]:
        print(f"• {c['name']}  [{c['type']}]")
        print(f"    stats:   {c.get('stats') or '(none)'}")
        print(f"    desc:    {(c['description'] or '(none)')[:80]}")
        print(f"    samples: {c['samples'][:5]}")
else:
    print("No datasets in the store yet — run the fetch cells above.")

## 8. Description inventory & split preview

How many usable **column descriptions** are in the store, and how they fall across the train / val / test split the finetune & eval notebooks build. The split is **held out by dataset** (a dataset's columns never straddle two splits) using the **same seed and fractions** as `finetune_descriptions.ipynb`, so these are the exact example counts training and evaluation will see — one `column_description` example per column that carries a usable description. (Columns with empty descriptions contribute nothing and are already excluded.)

In [ ]:
import random

# Must match finetune_descriptions.ipynb (seed + fractions) so the preview equals reality.
SEED, VAL_FRAC, TEST_FRAC = 42, 0.10, 0.10

store = load_all()


def _n_column_examples(rec):
    return sum(
        1
        for c in rec.get("columns") or []
        if len((c.get("description") or "").strip()) >= MIN_COLUMN_DESC_CHARS
    )


def _split_uids(uids, seed=SEED, val_frac=VAL_FRAC, test_frac=TEST_FRAC):
    """Deterministic held-out-by-dataset split (identical logic to the finetune notebook)."""
    uids = sorted(uids)
    random.Random(seed).shuffle(uids)
    n = len(uids)
    n_test = max(1, round(n * test_frac))
    n_val = max(1, round(n * val_frac))
    return {
        "train": uids[n_test + n_val :],
        "val": uids[n_test : n_test + n_val],
        "test": uids[:n_test],
    }


splits = _split_uids(list(store.keys()))

# whole-store totals
tot_col = sum(_n_column_examples(r) for r in store.values())
n_with_col = sum(1 for r in store.values() if _n_column_examples(r) > 0)
print(f"Datasets in store                : {len(store)}")
print(f"Datasets with >=1 column desc.   : {n_with_col}")
print(f"Usable column descriptions       : {tot_col}")
print(f"Total training examples          : {tot_col}\n")

# per-split breakdown
print(f"{'split':>6} | {'datasets':>8} | {'column_desc':>11}")
print("-" * 32)
for name in ("train", "val", "test"):
    uids = splits[name]
    n_col = sum(_n_column_examples(store[u]) for u in uids)
    print(f"{name:>6} | {len(uids):>8} | {n_col:>11}")